# Compare all comparable v2a-RSN connectivity methods

This notebook compares inferred connectivity estimates from c-GC, c-GC*, and
fixed-lag PCMCI+.

All three methods must contain the same four recordings, the same deterministic
`n25-e9-r16` trace selections, and `p=1,...,7`. Each method varies
conditioning complexity while holding its temporal lag configuration fixed.
PCMCI+ uses `tau_min=tau_max=1`, with `p` applied to its conditioning caps.

Outputs are written under
`outputs/v2a-RSNs/n25-e9-r16/method_comparison/`.


In [ ]:
from __future__ import annotations

import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PACKAGE_RELATIVE_PATH = Path('src/markovianity_diagnostic')
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / PACKAGE_RELATIVE_PATH).exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{PACKAGE_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from markovianity_diagnostic.experiments.v2a_rsn_utils import (  # noqa: E402
    V2A_ANALYSIS_PROFILE,
)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
EXPECTED_RECORDINGS = {
    '220119_F2_run11',
    '220127_F4_run2',
    '220210_F1_run6',
    '220210_F2_run5',
}

METHOD_SPECS = {
    'c-GC': {
        'directory': 'c-GC',
        'depth_semantics': 'conditioning-set depth',
        'color': '#1f77b4',
        'marker': 'o',
        'linestyle': '-',
    },
    'c-GC*': {
        'directory': 'c-GC-star',
        'depth_semantics': 'conditioning-set depth',
        'color': '#ff7f0e',
        'marker': 's',
        'linestyle': '--',
    },
    'PCMCI+': {
        'directory': 'pcmciplus',
        'depth_semantics': 'maximum conditioning-set size at fixed lag 1',
        'color': '#2ca02c',
        'marker': '^',
        'linestyle': '-.',
    },
}

V2A_OUTPUT_DIR = (
    PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / V2A_ANALYSIS_PROFILE
)
OUTPUT_DIR = V2A_OUTPUT_DIR / 'method_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Analysis profile: {V2A_ANALYSIS_PROFILE}')
print(f'Comparison depths: {P_VALUES}')
print(f'Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}')

In [ ]:
method_frames = {}
input_paths = []
reference_selection = {}

for method_label, spec in METHOD_SPECS.items():
    method_dir = V2A_OUTPUT_DIR / spec['directory']
    transitions_path = method_dir / 'transitions.csv'
    if not transitions_path.exists():
        raise FileNotFoundError(
            f'Missing {method_label} transitions: {transitions_path}'
        )

    frame = pd.read_csv(transitions_path)
    required_columns = {'dataset', 'P', 'edge_count', 'D_p', 'D_minus', 'D_plus'}
    missing_columns = sorted(required_columns.difference(frame.columns))
    if missing_columns:
        raise ValueError(
            f'{method_label} transitions lack columns: {missing_columns}'
        )

    frame['P'] = frame['P'].astype(int)
    recordings = set(frame['dataset'].unique())
    if recordings != EXPECTED_RECORDINGS:
        raise ValueError(
            'All methods must contain the same recordings; '
            f'{method_label} has {sorted(recordings)}, '
            f'expected {sorted(EXPECTED_RECORDINGS)}'
        )

    for recording in sorted(EXPECTED_RECORDINGS):
        recording_depths = set(
            frame.loc[frame['dataset'] == recording, 'P'].astype(int)
        )
        if not set(P_VALUES).issubset(recording_depths):
            raise ValueError(
                'All methods must contain P=1,...,7 for every recording; '
                f'{method_label}/{recording} has {sorted(recording_depths)}'
            )

        metadata_path = method_dir / recording / 'run_metadata.json'
        if not metadata_path.exists():
            raise FileNotFoundError(
                f'Missing selection metadata for {method_label}/{recording}: '
                f'{metadata_path}'
            )
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        if metadata.get('analysis_profile') != V2A_ANALYSIS_PROFILE:
            raise ValueError(
                f'{method_label}/{recording} was not generated with '
                f'{V2A_ANALYSIS_PROFILE}'
            )
        selected_indices = metadata.get('trace_selection', {}).get(
            'selected_cell_indices'
        )
        if not isinstance(selected_indices, list) or len(selected_indices) != 25:
            raise ValueError(
                f'{method_label}/{recording} lacks a valid 25-trace selection'
            )
        if recording in reference_selection:
            if selected_indices != reference_selection[recording]:
                raise ValueError(
                    f'Trace selection differs across methods for {recording}'
                )
        else:
            reference_selection[recording] = selected_indices
        input_paths.extend([transitions_path, metadata_path])

    frame = frame.loc[frame['P'].isin(P_VALUES)].copy()
    duplicate_rows = frame.duplicated(['dataset', 'P'])
    if duplicate_rows.any():
        duplicates = frame.loc[duplicate_rows, ['dataset', 'P']].to_dict('records')
        raise ValueError(f'Duplicate {method_label} rows: {duplicates}')

    frame['method'] = method_label
    frame['method_directory'] = spec['directory']
    frame['depth_semantics'] = spec['depth_semantics']
    method_frames[method_label] = frame
    print(
        f'Loaded {method_label}: {len(frame)} rows, '
        f'{len(recordings)} recordings'
    )

all_data = pd.concat(method_frames.values(), ignore_index=True)
recording_labels = {
    recording: f'fish-{index}'
    for index, recording in enumerate(sorted(EXPECTED_RECORDINGS), start=1)
}
all_data['recording_label'] = all_data['dataset'].map(recording_labels)

print('Validated identical 25-trace selections across all three methods.')

In [ ]:
summary_rows = []
for (method, recording), group in all_data.groupby(['method', 'dataset']):
    group = group.sort_values('P')
    instability = group.dropna(subset=['D_p'])
    edge_p1 = int(group.loc[group['P'] == 1, 'edge_count'].iloc[0])
    edge_p7 = int(group.loc[group['P'] == 7, 'edge_count'].iloc[0])
    summary_rows.append({
        'method': method,
        'method_directory': METHOD_SPECS[method]['directory'],
        'depth_semantics': METHOD_SPECS[method]['depth_semantics'],
        'recording': recording,
        'recording_label': recording_labels[recording],
        'edge_count_p1': edge_p1,
        'edge_count_p7': edge_p7,
        'edge_count_change': edge_p7 - edge_p1,
        'edge_count_change_fraction': (
            (edge_p7 - edge_p1) / edge_p1 if edge_p1 else np.nan
        ),
        'mean_edge_count': float(group['edge_count'].mean()),
        'max_D_p': float(instability['D_p'].max()),
        'max_D_p_depth': int(
            instability.loc[instability['D_p'].idxmax(), 'P']
        ),
        'cumulative_D_p': float(instability['D_p'].sum()),
        'mean_D_minus': float(instability['D_minus'].mean()),
        'mean_D_plus': float(instability['D_plus'].mean()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ['recording', 'method']
).reset_index(drop=True)
summary_path = OUTPUT_DIR / 'method_comparison_summary.csv'
summary_df.to_csv(summary_path, index=False)

print(summary_df.to_string(index=False))
print(f'Wrote {summary_path.relative_to(PROJECT_ROOT)}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)

for axis, recording in zip(axes.ravel(), sorted(EXPECTED_RECORDINGS)):
    for method, spec in METHOD_SPECS.items():
        values = all_data[
            (all_data['dataset'] == recording)
            & (all_data['method'] == method)
        ].sort_values('P')
        axis.plot(
            values['P'],
            values['edge_count'],
            color=spec['color'],
            marker=spec['marker'],
            linestyle=spec['linestyle'],
            linewidth=1.8,
            markersize=5,
            label=method,
        )
    axis.set_title(recording_labels[recording])
    axis.set_xlabel('Reported depth index p')
    axis.set_ylabel('Directed edge count')
    axis.set_xticks(P_VALUES)
    axis.grid(alpha=0.3)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3)
fig.text(
    0.5,
    0.01,
    'All methods: increasing conditioning complexity with fixed temporal lag settings',
    ha='center',
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.035, 1, 0.94))
edge_plot_path = OUTPUT_DIR / 'edge_count_comparison.png'
fig.savefig(edge_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Wrote {edge_plot_path.relative_to(PROJECT_ROOT)}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)

for axis, recording in zip(axes.ravel(), sorted(EXPECTED_RECORDINGS)):
    for method, spec in METHOD_SPECS.items():
        values = all_data[
            (all_data['dataset'] == recording)
            & (all_data['method'] == method)
        ].dropna(subset=['D_p']).sort_values('P')
        axis.plot(
            values['P'],
            values['D_p'],
            color=spec['color'],
            marker=spec['marker'],
            linestyle=spec['linestyle'],
            linewidth=1.8,
            markersize=5,
            label=method,
        )
    axis.set_title(recording_labels[recording])
    axis.set_xlabel('Reported depth index p')
    axis.set_ylabel('Adjacent-depth instability D_p')
    axis.set_xticks(P_VALUES[1:])
    axis.grid(alpha=0.3)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3)
fig.text(
    0.5,
    0.01,
    'PCMCI+ p is a maximum conditioning limit; c-GC p remains its native conditioning depth.',
    ha='center',
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.035, 1, 0.94))
instability_plot_path = OUTPUT_DIR / 'instability_comparison.png'
fig.savefig(instability_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Wrote {instability_plot_path.relative_to(PROJECT_ROOT)}')

In [ ]:
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'v2a three-method inferred-connectivity comparison',
    'analysis_profile': V2A_ANALYSIS_PROFILE,
    'methods': list(METHOD_SPECS),
    'method_directories': {
        method: spec['directory'] for method, spec in METHOD_SPECS.items()
    },
    'depths': P_VALUES,
    'depth_semantics': {
        method: spec['depth_semantics']
        for method, spec in METHOD_SPECS.items()
    },
    'recordings': sorted(EXPECTED_RECORDINGS),
    'trace_selection_contract': (
        'identical ordered n25-e9-r16 indices across all methods'
    ),
    'interpretation_limit': (
        'all methods vary conditioning complexity; PCMCI+ uses fixed lag 1'
    ),
    'input_paths': sorted(
        {str(path.relative_to(PROJECT_ROOT)) for path in input_paths}
    ),
    'output_paths': [
        str(summary_path.relative_to(PROJECT_ROOT)),
        str(edge_plot_path.relative_to(PROJECT_ROOT)),
        str(instability_plot_path.relative_to(PROJECT_ROOT)),
    ],
    'software_versions': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'matplotlib': plt.matplotlib.__version__,
    },
}
manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

expected_outputs = [
    summary_path,
    edge_plot_path,
    instability_plot_path,
    manifest_path,
]
missing_outputs = [path for path in expected_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Missing comparison outputs: {missing_outputs}')

expected_rows = len(METHOD_SPECS) * len(EXPECTED_RECORDINGS)
assert len(summary_df) == expected_rows
assert set(all_data['P']) == set(P_VALUES)
assert set(all_data['method']) == set(METHOD_SPECS)
print(
    f'Verified {len(expected_outputs)} outputs and '
    f'{expected_rows} method-recording summaries.'
)